In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

tidy = pd.read_csv("data/tidy.csv")
np.random.seed(0)

In [ ]:
def get(construct, variant, experiment, condition):
    m = ((tidy.construct == construct) & (tidy.variant == variant) &
         (tidy.experiment == experiment) & (tidy.condition == condition))
    return tidy.loc[m, "pct_change"]

In [ ]:
def box_points(ax, data, title):
    ax.boxplot(list(data.values()), tick_labels=list(data.keys()),
               whis=(0, 100), showfliers=False)
    for i, vals in enumerate(data.values(), start=1):
        ax.scatter(np.random.normal(i, 0.04, len(vals)), vals, s=18, alpha=0.7, zorder=3)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_title(title)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, c in zip(axes, ["GFP", "cpGFP"]):
    data = {"55 mmHg": get(c, "WT", "response", "55"),
            "Depol": get(c, "WT", "response", "depol")}
    box_points(ax, data, f"Cx26-{c}")
axes[0].set_ylabel("Fluorescence change (%)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, c in zip(axes, ["GFP", "cpGFP"]):
    data = {"35 mmHg": get(c, "WT", "DR", "35"),
            "55 mmHg": get(c, "WT", "DR", "55"),
            "70 mmHg": get(c, "WT", "DR", "70")}
    box_points(ax, data, f"Cx26-{c}")
axes[0].set_ylabel("Fluorescence change (%)")
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

comparisons = [
    ("GFP WT vs K125R",   ("GFP", "WT"),   ("GFP", "K125")),
    ("cpGFP WT vs K125R", ("cpGFP", "WT"), ("cpGFP", "K125")),
    ("GFP WT vs LL",      ("GFP", "WT"),   ("GFP", "LL")),
]

fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=True)
for ax, (title, a, b) in zip(axes, comparisons):
    x = get(a[0], a[1], "response", "55")
    y = get(b[0], b[1], "response", "55")
    p = stats.mannwhitneyu(x, y, alternative="two-sided", method="exact").pvalue

    data = {
        f"{a[1]}\n(n={len(x)})": x,
        f"{b[1]}\n(n={len(y)})": y,
    }
    box_points(ax, data, f"{title}\np = {p:.3f}")

axes[0].set_ylabel("Fluorescence change (%)")
plt.tight_layout()
plt.show()